# Notebook 09 — Final Multi-Task Model
## Integrating Category Hierarchy, Priority, and Sentiment

**Project**: Cloud-Based ITSM Ticket Classification Platform Using Fine-Tuned Transformer Models  
**Author**: Mohamed A. Elbaz  

---

### Objective

This notebook represents the **final production-grade model**. We integrate all five tasks into a single multi-task MarBERTv2 architecture:
1. **Category L1** (6 classes)
2. **Category L2** (16 classes)
3. **Category L3** (48 classes)
4. **Priority** (5 classes)
5. **Sentiment** (4 classes)

**Workflow**:
1. Load the full processed dataset with all 5 task labels.
2. Fine-tune with 5 independent classification heads.
3. Perform a comprehensive evaluation of all tasks.
4. Save the final production model for deployment (Inference/API).

> **Note on MarBERT Load Report**: When loading the model, you may see "UNEXPECTED" keys in the load report (e.g., `cls.predictions.*`). This is **normal behavior**. MarBERTv2 was pre-trained with Masked Language Modeling (MLM) and Next Sentence Prediction (NSP) heads. Since we are using it for classification, these heads are discarded and replaced by our custom task heads. You can safely ignore these warnings.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import pandas as pd
import pickle
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

from arabic_itsm.data.preprocessing import ArabicTextNormalizer
from arabic_itsm.data.dataset import ITSMDataset
from arabic_itsm.models.classifier import MarBERTClassifier
from arabic_itsm.utils.metrics import compute_classification_metrics

MODEL_NAME = "UBC-NLP/MARBERTv2"
TASKS      = ['l1', 'l2', 'l3', 'priority', 'sentiment']
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIG_DIR    = Path('../results/figures')
METRICS_DIR = Path('../results/metrics')

## 1. Full Dataset Preparation

In [ ]:
DATA_DIR = Path('../data/processed')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')

with open(DATA_DIR / 'label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
normalizer = ArabicTextNormalizer()

train_ds = ITSMDataset(train_df, tokenizer, normalizer, label_encoders, tasks=TASKS)
val_ds   = ITSMDataset(val_df, tokenizer, normalizer, label_encoders, tasks=TASKS)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"Final Task Set: {TASKS}")
print(f"Class Counts: {train_ds.num_classes}")

## 2. Final Multi-Task Training Loop

In [ ]:
model = MarBERTClassifier(MODEL_NAME, train_ds.num_classes).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

best_avg_f1 = 0.0
OUT_DIR = Path('../models/marbert_final_multitask')
OUT_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(1, 6):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = {f"label_{t}": batch[f"label_{t}"].to(DEVICE) for t in TASKS}

        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            out = model(ids, mask, **labels)
            loss = out["loss"]

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    # Validation
    model.eval()
    all_preds = {t: [] for t in TASKS}
    all_labels = {t: [] for t in TASKS}
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            out = model(ids, mask)
            for t in TASKS:
                all_preds[t].extend(torch.argmax(out[f"logits_{t}"], dim=-1).cpu().numpy())
                all_labels[t].extend(batch[f"label_{t}"].numpy())

    # Compute Metrics per Task
    f1_scores = []
    print(f"Epoch {epoch} Results:")
    for t in TASKS:
        res = compute_classification_metrics(all_labels[t], all_preds[t])
        f1_scores.append(res['macro_f1'])
        print(f"  {t:<10}: Macro-F1 = {res['macro_f1']:.4f}")
    
    avg_f1 = np.mean(f1_scores)
    print(f"Average Macro-F1 across all tasks: {avg_f1:.4f}")

    if avg_f1 > best_avg_f1:
        best_avg_f1 = avg_f1
        torch.save(model.heads.state_dict(), OUT_DIR / "heads.pt")
        model.encoder.save_pretrained(str(OUT_DIR))
        print("  ✓ Saved new best full multi-task checkpoint")

## 3. Comprehensive Evaluation Visualization
We compare the final performance across all tasks to see where the model is most reliable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(x=TASKS, y=f1_scores, palette='coolwarm')
plt.ylim(0, 1)
plt.title('Final Multi-Task Model Performance (Macro-F1)', fontweight='bold')
plt.ylabel('Macro-F1 Score')
for i, v in enumerate(f1_scores):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontweight='bold')
plt.savefig(FIG_DIR / '09_final_multitask_performance.png', bbox_inches='tight')
plt.show()

print("\n--- Final Summary ---")
print("The model now successfully predicts Category (L1/L2/L3), Priority, and Sentiment in a single pass.")
print("Typically, L1 and Sentiment perform best, while L3 and Priority are the most difficult due to class imbalance and label subtlety.")

## 4. Conclusion & Next Phase

We have completed the experimental phase of the project:
1. **Baselines & Signifance** established.
2. **Stepwise taxonomy expansion** (L1 → L2 → L3) successful.
3. **Auxiliary tasks** (Priority & Sentiment) integrated.

**The next phase of the project will focus on Deployment and Explainability.**